# 第四章：LangChain Neo4j 集成

## 学习目标

- 使用 Neo4jGraph 简化图数据库操作
- 理解 Schema 自动提取的作用
- 用 GraphCypherQAChain 实现自然语言图问答
- 使用 Neo4jVector 在图上做向量检索
- 对比手写 Cypher vs LLM 生成 Cypher

## 0. 环境准备

加载环境变量，连接 Neo4j，初始化 LLM，并准备电影数据集。

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

# 切换为 GLM（同样是 OpenAI 兼容接口）：
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="glm-4-plus", openai_api_base=os.environ["GLM_API_BASE"], openai_api_key=os.environ["GLM_API_KEY"])

In [ ]:
# 先用原生驱动准备数据
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
)
driver.verify_connectivity()
print("\u2713 Neo4j 连接成功")

In [ ]:
# 清空数据库，从零开始
driver.execute_query("MATCH (n) DETACH DELETE n")
print("\u2713 数据库已清空")

In [ ]:
# 创建电影数据集（与第二章相同）
driver.execute_query("""
    CREATE (inception:Movie {title: '盗梦空间', year: 2010, genre: '科幻'})
    CREATE (matrix:Movie {title: '黑客帝国', year: 1999, genre: '科幻'})
    CREATE (interstellar:Movie {title: '星际穿越', year: 2014, genre: '科幻'})
    CREATE (titanic:Movie {title: '泰坦尼克号', year: 1997, genre: '爱情'})
    CREATE (godfather:Movie {title: '教父', year: 1972, genre: '犯罪'})
    
    CREATE (nolan:Person {name: '克里斯托弗·诺兰', born: 1970})
    CREATE (leo:Person {name: '莱昂纳多·迪卡普里奥', born: 1974})
    CREATE (keanu:Person {name: '基努·里维斯', born: 1964})
    CREATE (anne:Person {name: '安妮·海瑟薇', born: 1982})
    CREATE (brando:Person {name: '马龙·白兰度', born: 1924})
    
    CREATE (nolan)-[:DIRECTED]->(inception)
    CREATE (nolan)-[:DIRECTED]->(interstellar)
    CREATE (leo)-[:ACTED_IN {role: 'Cobb'}]->(inception)
    CREATE (leo)-[:ACTED_IN {role: 'Jack'}]->(titanic)
    CREATE (keanu)-[:ACTED_IN {role: 'Neo'}]->(matrix)
    CREATE (anne)-[:ACTED_IN {role: 'Brand'}]->(interstellar)
    CREATE (anne)-[:ACTED_IN {role: 'Selina'}]->(inception)
    CREATE (brando)-[:ACTED_IN {role: 'Vito Corleone'}]->(godfather)
    
    CREATE (keanu)-[:FRIENDS_WITH]->(leo)
    CREATE (leo)-[:FRIENDS_WITH]->(anne)
""")
print("\u2713 电影数据集创建完成")

In [ ]:
# 验证数据
records, _, _ = driver.execute_query("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(n) AS count
""")
for r in records:
    print(f"  {r['label']}: {r['count']} 个节点")

# 关闭原生驱动（后续使用 LangChain 的封装）
driver.close()
print("\n\u2713 原生驱动已关闭，接下来使用 LangChain 封装")

### 刚才发生了什么？

我们做了三件事：
1. 加载环境变量，初始化 Qwen LLM
2. 用原生 `neo4j` 驱动连接数据库并写入电影数据集（5 部电影、5 个人物、多条关系）
3. 关闭原生驱动——从下一节开始，我们将使用 LangChain 提供的 `Neo4jGraph` 封装，不再直接操作原生驱动

> 这个电影数据集和第二章中使用的相同，包含 `Movie`、`Person` 节点以及 `DIRECTED`、`ACTED_IN`、`FRIENDS_WITH` 关系。

## 1. Neo4jGraph：统一的图操作接口

在前三章中，我们一直使用原生的 `neo4j` Python 驱动。LangChain 生态提供了 `langchain-neo4j` 包，其中的 `Neo4jGraph` 类封装了驱动，提供了更简洁的接口。

In [ ]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)
print("\u2713 Neo4jGraph 初始化成功")

In [ ]:
# 便捷查询
result = graph.query("MATCH (m:Movie) RETURN m.title AS title LIMIT 3")
for r in result:
    print(f"  {r['title']}")

### 刚才发生了什么？

`Neo4jGraph` 封装了原生驱动，提供了更简单的接口：

| | 原生驱动 | Neo4jGraph |
|---|---|---|
| **初始化** | `GraphDatabase.driver(uri, auth=(...))` | `Neo4jGraph(url=..., username=..., password=...)` |
| **查询** | `driver.execute_query(cypher)` 返回 `(records, summary, keys)` | `graph.query(cypher)` 直接返回结果列表 |
| **返回值** | 三元组，需要解包 | 字典列表，直接可用 |
| **额外功能** | 无 | 自动提取 Schema（下一节讲） |

`graph.query()` 省去了解包三元组的步骤，返回的直接就是字典列表——和 `records` 等效，但更简洁。

## 2. Schema 自动提取

`Neo4jGraph` 最强大的功能之一是**自动提取图的 Schema**。这个 Schema 描述了图中有哪些节点标签、关系类型、以及它们的属性——这对后面的 LLM 查询至关重要。

In [ ]:
# 查看自动提取的图 Schema
print(graph.schema)

### 刚才发生了什么？

`graph.schema` 返回一个字符串，描述了图的完整结构：

- **节点标签**：`Movie`、`Person` 等，以及它们各自的属性（`title`、`year`、`genre`、`name`、`born` 等）
- **关系类型**：`DIRECTED`、`ACTED_IN`、`FRIENDS_WITH` 等，以及起止节点和属性
- **属性的数据类型**：字符串、整数等

**为什么 Schema 重要？** 因为当我们让 LLM 根据自然语言生成 Cypher 时，LLM 需要知道：
- 图里有哪些标签（才能写出正确的 `MATCH (n:Movie)` 而不是 `MATCH (n:Film)`）
- 节点有哪些属性（才能用 `m.title` 而不是 `m.name` 来访问电影名称）
- 关系的方向和类型（才能写出正确的模式）

> Schema 在 `Neo4jGraph` 初始化时自动提取。如果图结构发生变化，可以调用 `graph.refresh_schema()` 手动刷新。

## 3. 对比：手写 Cypher vs 自然语言查询

前三章中，每次查询都需要手写 Cypher。如果我们能用自然语言提问，让 LLM 自动生成 Cypher 并执行，那就真正打通了「不懂 Cypher 也能查图」的路径。

In [ ]:
# \u274c 手写 Cypher（需要学 Cypher 语法）
result = graph.query("""
    MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
    WHERE m.title = '盗梦空间'
    RETURN p.name AS actor
""")
print("手写 Cypher 结果:")
for r in result:
    print(f"  {r['actor']}")

In [ ]:
# \u2705 用自然语言提问（LLM 自动生成 Cypher）
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,  # 显示生成的 Cypher
    allow_dangerous_requests=True,
)

response = chain.invoke({"query": "盗梦空间的演员是谁？"})
print("\n自然语言查询结果:")
print(response["result"])

### 刚才发生了什么？

两段代码回答了同一个问题：「盗梦空间的演员是谁？」

- **手写 Cypher**：你需要知道 `ACTED_IN` 关系的方向、`Movie` 标签、`title` 属性名……这些都需要 Cypher 知识
- **GraphCypherQAChain**：你只需要用自然语言提问，LLM 会自动：
  1. 读取 `graph.schema`，了解图的结构
  2. 根据问题生成 Cypher 查询
  3. 在 Neo4j 上执行 Cypher
  4. 将查询结果 + 原始问题交给 LLM，生成自然语言回答

`verbose=True` 让你在控制台看到中间过程（生成的 Cypher、查询结果等），方便调试。

> `allow_dangerous_requests=True` 是必须的——它明确告知你理解 LLM 可能生成修改数据的 Cypher。生产环境应使用只读数据库用户。

## 4. Text-to-Cypher 的工作原理

`GraphCypherQAChain` 内部是一个两步 LLM 流水线：

```
用户问题 + 图 Schema
        \u2193
   [LLM 第一次调用] \u2192 生成 Cypher
        \u2193
   [Neo4j 执行 Cypher] \u2192 返回原始数据
        \u2193
   用户问题 + 原始数据
        \u2193
   [LLM 第二次调用] \u2192 生成自然语言回答
        \u2193
      最终回答
```

两次 LLM 调用各司其职：第一次专注于「把自然语言翻译成 Cypher」，第二次专注于「把查询结果翻译成人话」。

让我们用多个问题来测试这个流水线。

In [ ]:
# 多个问题测试
questions = [
    "哪些电影是科幻片？",
    "克里斯托弗·诺兰导演了哪些电影？",
    "基努·里维斯和谁是朋友？",
]

for q in questions:
    print(f"\nQ: {q}")
    response = chain.invoke({"query": q})
    print(f"A: {response['result']}")
    print("-" * 40)

### 刚才发生了什么？

三个问题覆盖了不同的查询模式：

| 问题 | 涉及的模式 |
|---|---|
| 哪些电影是科幻片？ | 属性过滤：`WHERE m.genre = '科幻'` |
| 诺兰导演了哪些电影？ | 关系遍历：`(:Person)-[:DIRECTED]->(:Movie)` |
| 基努·里维斯和谁是朋友？ | 关系遍历：`(:Person)-[:FRIENDS_WITH]->(:Person)` |

注意观察 `verbose=True` 输出的 Cypher——LLM 生成的查询是否和你手写的一样？有时 LLM 会生成更冗长但等效的 Cypher，有时可能生成更优雅的写法。

## 5. 调试：查看生成的 Cypher

`verbose=True` 会在控制台打印中间过程，但如果你想在代码中获取生成的 Cypher 和查询结果，可以使用 `return_intermediate_steps=True`。

In [ ]:
# 通过 return_intermediate_steps 获取中间结果
chain_debug = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=False,
    return_intermediate_steps=True,
    allow_dangerous_requests=True,
)

response = chain_debug.invoke({"query": "2010年之后的科幻电影有哪些？"})

print("生成的 Cypher:")
print(response["intermediate_steps"][0]["query"])
print("\n查询结果:")
print(response["intermediate_steps"][1]["context"])
print("\n最终回答:")
print(response["result"])

### 刚才发生了什么？

`return_intermediate_steps=True` 让返回值多了一个 `intermediate_steps` 字段，包含：

- `intermediate_steps[0]["query"]`：LLM 生成的 Cypher 语句
- `intermediate_steps[1]["context"]`：Cypher 在 Neo4j 上执行后的原始结果

这对调试非常有用：
- 如果最终回答不对，先看生成的 Cypher 是否正确
- 如果 Cypher 正确但回答不对，说明是第二步 LLM 的问题
- 如果 Cypher 不正确，可以检查 Schema 是否完整、问题表述是否清晰

### 常见问题

- **生成的 Cypher 语法错误**：LLM 可能生成无效 Cypher。可以通过 `validate_cypher=True` 参数启用语法验证。
- **属性名不匹配**：LLM 可能猜错属性名（如用 `name` 而非 `title`）。Schema 的质量直接影响生成质量。
- **复杂查询准确率低**：多跳关系查询的准确率明显低于简单查询。可以提供示例（Few-shot）来提升。
- **安全风险**：LLM 可能生成 `DELETE`/`DROP` 等危险语句。生产环境务必使用只读权限的数据库用户。
- **`allow_dangerous_requests=True`**：这个参数是安全确认，表明你了解 LLM 可能生成修改数据的语句。不设置会报错。

## 6. Neo4jVector：图上的向量检索

在 LangChain 第五章中，我们用 FAISS 做了向量检索。`Neo4jVector` 可以把向量嵌入直接存储为 Neo4j 节点的属性，让向量检索和图遍历在同一个数据库中完成。

In [ ]:
from langchain_neo4j import Neo4jVector
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-v3",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

print("\u2713 Embeddings 模型初始化成功")

In [ ]:
# 从已有图中创建向量索引（基于节点的文本属性）
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    node_label="Movie",
    text_node_properties=["title", "genre"],
    embedding_node_property="embedding",
)

print("\u2713 Neo4jVector 索引创建完成")

### 刚才发生了什么？

`Neo4jVector.from_existing_graph()` 做了以下事情：

1. 找到所有 `Movie` 标签的节点
2. 拼接每个节点的 `title` 和 `genre` 属性文本
3. 调用 Embedding 模型生成向量
4. 把向量存储到每个节点的 `embedding` 属性中
5. 在 Neo4j 中创建向量索引，支持相似度搜索

关键参数：

| 参数 | 说明 |
|---|---|
| `node_label` | 对哪种标签的节点建索引 |
| `text_node_properties` | 用哪些文本属性生成向量 |
| `embedding_node_property` | 向量存储到节点的哪个属性中 |
| `embedding` | 使用的 Embedding 模型 |

In [ ]:
# 向量相似度搜索
results = vector_index.similarity_search("科幻动作片", k=3)
for doc in results:
    print(f"  {doc.page_content}")

### 刚才发生了什么？

`similarity_search()` 把查询文本「科幻动作片」转成向量，然后在 Neo4j 的向量索引中找到最相似的 3 个 Movie 节点。返回的 `Document` 对象的 `page_content` 就是我们指定的 `text_node_properties` 拼接出的文本。

这和 FAISS 的 `similarity_search()` 接口完全一致——LangChain 的向量存储抽象让你可以在不同后端之间无缝切换。

## 7. 对比：FAISS vs Neo4jVector

| | FAISS | Neo4jVector |
|---|---|---|
| **存储位置** | 独立文件 | Neo4j 数据库内 |
| **元数据** | `Document.metadata` 字典 | 节点属性 |
| **关联查询** | 不支持 | 可结合图遍历 |
| **适合场景** | 纯文本检索 | 需要结合关系信息的检索 |
| **已学章节** | LangChain 第五章 | 本章 |

**Neo4jVector 的核心优势**：向量作为节点属性存储，意味着你可以在一次查询中同时使用向量相似度和图遍历。例如：

```cypher
// 先用向量搜索找到相似电影，再沿关系找到导演
CALL db.index.vector.queryNodes('vector_index', 3, $embedding)
YIELD node AS movie, score
MATCH (d:Person)-[:DIRECTED]->(movie)
RETURN movie.title, d.name, score
```

这种「向量 + 图」的组合查询是纯向量存储（FAISS）无法做到的。

## 8. 实战：电影知识库自然语言问答

把前面学到的 `GraphCypherQAChain` 用到实际场景：一个电影知识库的自然语言问答系统。

In [ ]:
# 综合问答
demo_questions = [
    "有哪些科幻电影？列出名称和年份。",
    "莱昂纳多演过哪些电影？",
    "谁导演了盗梦空间？",
    "2000年以前的电影有哪些？",
]

print("=== 电影知识库问答 ===\n")
for q in demo_questions:
    response = chain.invoke({"query": q})
    print(f"Q: {q}")
    print(f"A: {response['result']}\n")

### 刚才发生了什么？

四个问题覆盖了多种查询场景：

| 问题 | 查询类型 |
|---|---|
| 科幻电影的名称和年份 | 属性过滤 + 多字段返回 |
| 莱昂纳多演过的电影 | 关系遍历（模糊匹配人名） |
| 谁导演了盗梦空间 | 反向关系遍历 |
| 2000年以前的电影 | 数值比较过滤 |

注意 LLM 处理「莱昂纳多」时的表现——它需要理解这是「莱昂纳多·迪卡普里奥」的简称。如果匹配失败，可能是因为 LLM 生成了精确匹配的 Cypher（`WHERE p.name = '莱昂纳多'`）而不是模糊匹配（`WHERE p.name CONTAINS '莱昂纳多'`）。这也是 Text-to-Cypher 的局限之一。

## 总结

本章学习了：

- \u2705 `Neo4jGraph` 简化图操作，自动提取 Schema
- \u2705 `GraphCypherQAChain` 实现 Text-to-Cypher 自然语言图问答
- \u2705 调试技巧：查看生成的 Cypher 和中间结果
- \u2705 `Neo4jVector` 在图上做向量检索
- \u2705 FAISS vs Neo4jVector 的适用场景
- \u2705 对比：手写 Cypher vs LLM 自动生成

### 核心收获

| 工具 | 用途 |
|---|---|
| `Neo4jGraph` | 统一的图操作接口，自动提取 Schema |
| `GraphCypherQAChain` | 自然语言 \u2192 Cypher \u2192 自然语言回答 |
| `Neo4jVector` | 在 Neo4j 中存储和检索向量 |

这三个组件构成了 LangChain 与 Neo4j 集成的核心。下一章我们将进一步探索如何从非结构化文本中自动构建知识图谱。

## 下一章预告

**第五章：LLM 知识图谱自动构建** —— 本章能查询已有的图数据，但图本身还是手动构建的。下一章学习用 `LLMGraphTransformer` 从非结构化文本中自动提取实体和关系，构建知识图谱。